In [1]:
import os
import pandas as pd

In [2]:
with open('/ptmp/rfechner/out/exp05_rollouts_qwen2.5-7b/qwen2.5_7b__kl-cov/val_jsonl/0_rollouts.jsonl') as file:
    base_df = pd.read_json(file, lines=True)

In [3]:
with open('/ptmp/rfechner/out/exp05_rollouts_qwen2.5-7b/qwen2.5_7b__kl-cov/val_jsonl/60_rollouts.jsonl') as file:
    klcov_df = pd.read_json(file, lines=True)

In [4]:
with open('/ptmp/rfechner/out/exp05_rollouts_qwen2.5-7b/qwen2.5_7b__gspo/val_jsonl/80_rollouts.jsonl') as file:
    gspo_df = pd.read_json(file, lines=True)

In [5]:
klcov_df.columns

Index(['input', 'output', 'gts', 'score', 'step', 'reward', 'format_score',
       'acc', 'extracted_gt'],
      dtype='object')

In [6]:
base_df['method'] = ['base'] * len(base_df)
klcov_df['method'] = ['klcov'] * len(klcov_df)
gspo_df['method'] = ['gspo'] * len(gspo_df)
dfs = [base_df, klcov_df]

# subsample dataframes per-question
subdfs = []
for df in dfs:
    subdf = (
                df
                .groupby('input', sort=False, group_keys=False)
                .sample(n=64, replace=False, random_state=0)
            )
    subdfs.append(subdf)

df = pd.concat(subdfs, axis=0)

In [ ]:
def construct_prompt_verify(row : pd.DataFrame) -> list[dict]:
    system = "You are a helpful judge and an expert in mathematical reasoning."
    
    prefix = (
        "You're given a question and a students answer. Answer '#### yes' if and only if the answer "
        "contains explicit (verbalized) verification of any intermediate or the final result - no matter whether the answer is correct. "
        "This includes explicit re-iterating over answers "
        "to check their correctness, verification by plugging in the found result back into the equation to check the answer satisfies "
        "conditions etc. This may include student answers which contain 'Let's verify the answer' or 'Let's check the answer' etc."
        "\nHowever, this does not include step-by-step initial solution of the problem, we're only looking for verbalized verification after "
        "a solution was already computed. In particular, verification using programming languages like Python do NOT count as valid verification. "
        "Answer '#### no' if the answer doesn't contain a valid verification.\n"
    )
    remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
    question = row['input'].removeprefix(remove_prefix).removesuffix(remove_suffix)
    student = row['output']
    gt = row['gts']
    prompt = [{
        'role' : 'system',
        'content' : system
    },
    {
        'role' : 'user',
        'content' : f"{prefix}\n\nQuestion:\n\n{question}\n\nStudent's answer:\n\n{student}\n\nGround Truth Solution:\n\n{gt}\n\nPlease only answer either '#### yes' or '#### no'.\n"
    }]
    return prompt

to_verify = pd.DataFrame({
    'prompt' : df.apply(construct_prompt_verify, axis=1),
    'method' : df['method'],
    'reward' : df['reward']})

In [8]:
to_verify.iloc[0]['prompt']

[{'role': 'system',
  'content': 'You are a helpful judge and an expert in mathematical reasoning.'},
 {'role': 'user',
  'content': "You're given a question and a students answer. Answer '#### yes' if and only if the answer contains explicit (verbalized) verification of any intermediate or the final result - no matter whether the answer is correct. This includes explicit re-iterating over answers to check their correctness, verification by plugging in the found result back into the equation to check the answer satisfies conditions etc. This may include student answers which contain 'Let's verify the answer' or 'Let's check the answer' etc.\nHowever, this does not include step-by-step initial solution of the problem, we're only looking for verbalized verification after a solution was already computed. In particular, verification using programming languages like Python do NOT count as valid verification. Answer '#### no' if the answer doesn't contain a valid verification.\n\n\nQuestion:

In [9]:
print(to_verify.iloc[0]['prompt'][1]['content'])

You're given a question and a students answer. Answer '#### yes' if and only if the answer contains explicit (verbalized) verification of any intermediate or the final result - no matter whether the answer is correct. This includes explicit re-iterating over answers to check their correctness, verification by plugging in the found result back into the equation to check the answer satisfies conditions etc. This may include student answers which contain 'Let's verify the answer' or 'Let's check the answer' etc.
However, this does not include step-by-step initial solution of the problem, we're only looking for verbalized verification after a solution was already computed. In particular, verification using programming languages like Python do NOT count as valid verification. Answer '#### no' if the answer doesn't contain a valid verification.


Question:

Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ an

In [10]:
outpath = "/u/rfechner/data/verified_at_k"
os.makedirs(outpath, exist_ok=True)
with open(os.path.join(outpath, 'base_klcov_64samples_chats.parquet'), 'wb') as file:
    to_verify.to_parquet(file)